# Torun WAQ — verify trimmed communication files and rebuild FPV scenarios

This notebook re-runs, in one place, the checks and rebuild steps worked out while diagnosing the
trimmed (dry-cell-removed) Delft3D-WAQ setup for the Torun Reservoir FPV scenarios.

**What it does**

| Part | Purpose | Writes files? |
|---|---|---|
| A | Build/load the segment mapping `new_to_old` (27,904 → 10,999 ever-wet segments) from the **baseline** FLOW `.vol` | `new_to_old.npy` |
| B | Verify every baseline trimmed file against the untrimmed FLOW output; build the exchange mapping `exch_map` | `exch_map.npy` |
| C | Check radiation and wind forcing (`.bin`) for each scenario | – |
| D | Rebuild each scenario's trimmed hydrodynamics from its **own** FLOW output with the **baseline** mapping; optionally rebuild a radiation `.bin` with the intended shading factor | new `XXscen_fixed/` folders, new `.bin` |
| E | Audit existing clean folders: which FLOW run (and which segment order) were they built from? | – |
| F | Model diagnostics from a finished run (`RefDay`/day length, `Depth`, dry surface segment, `.lst` warnings) | – |
| G | `.inp` checklist and open issues | – |

**How to use**

1. Set `MODEL_DIR` and check the rest of the configuration cell (Section 1).
2. Run Sections 1–2 (configuration + helper functions) and Part A, then any Part you need. Part D additionally needs `exch_map.npy`, which Part B (B7) creates once.
3. Long checks read many GB over the network. They use `STEP` (check every `STEP`-th record). Use `STEP = 10` for a quick pass and `STEP = 1` for a full check (e.g. overnight).
4. Nothing existing is overwritten: rebuilds go to new folders/files. A summary of all PASS/FAIL results is printed at the end (Section H).

**File formats assumed** (all confirmed on the Torun files during the diagnosis unless marked otherwise)

* Segment files (`.vol`, `.tem`, `.vdf`, `.tau`, forcing `.bin`): records of `int32 time` + `nseg × float32`. Untrimmed `.vdf`/`.tau` end with a 4-byte partial record, which is dropped.
* Exchange files (`.flo`, `.are`): records of `int32 time` + `noq × float32`.
* `.poi`: `noq × 4` int32 (from, to, from−1, to+1); 0 = no segment. No open boundaries in this model (no negative pointers).
* `.len`: 4-byte header, then **all "from" lengths followed by all "to" lengths** (not interleaved).
* `.par` (`SURF`, `bottomdept`): 4-byte header, then values grouped **per parameter** (all `SURF`, then all `bottomdept`).
* `.srf` (untrimmed): 4-byte header + one value per segment — *assumed from the file size; it matched `SURF` exactly.*
* `.atr`: two blocks — attribute 1 (1 = active) and attribute 2 (vertical position). The code meanings 0 = surface+bottom, 1 = surface, 2 = middle, 3 = bottom are *my understanding of DELWAQ — verify in the D-WAQ manual.*
* `.his`: 160-char title, `nvar`, `nloc`, 20-char names, location id + 20-char name, then records `int32 time` + `nloc × nvar float32` — *from memory of the format; it reproduced `Temp` exactly at all stations.*

**Background (short)**: the trimmed setup removes always-dry cells so light starts at the first wet cell. The checks below found that
the baseline trimmed files are correct, but scenario 02 was built with its own 11,000-cell mask (segments shifted by one),
scenarios 03/04 used the baseline hydrodynamics, all scenarios used baseline `.flo`/`.are`, and 03/04 radiation used factor 0.10
instead of 0.65. Model-level issues independent of trimming are listed in Part G.

## 1. Configuration

Edit this cell only. Paths follow the folder layout on the network drive. `MODEL_DIR` is the folder `wq_0.1min_secondfolder`.

In [ ]:
import datetime as dt
from pathlib import Path

# ---- folders ----------------------------------------------------------------------------------
MODEL_DIR  = Path(r"<MODEL_DIR>")                 # e.g. ...\delft_modelling\wq_0.1min_secondfolder
RAW_DIR    = MODEL_DIR / "com_files"             # untrimmed FLOW communication files
BL_DIR     = MODEL_DIR / "01scen_baseline"       # baseline run folder (trimmed files actually used)
WORK_DIR   = MODEL_DIR / "trim_checks"           # mappings and small outputs of this notebook
FORCING_DIR = MODEL_DIR                          # folder holding the waq-*.bin forcing files
SEGNUM_MAT = MODEL_DIR / "segment number.mat"    # QUICKPLOT segment-number grid (M, N, K)

CLEAN      = "com-WQexport_clean"                # prefix of trimmed files inside a run folder
RAW_BASE   = RAW_DIR / "com-WQexport_30min_fullperiod"   # untrimmed baseline prefix (no extension)
BASE_INP   = BL_DIR / "Torun_WAQ_Apr2023.inp"
BASE_PAR   = BL_DIR / "Torun_WAQ_Apr2023.par"

# ---- model dimensions -------------------------------------------------------------------------
N_SEG_IN   = 27904        # untrimmed segments (1,744 columns x 16 z-layers)
N_SEG      = 10999        # ever-wet segments kept in the trimmed grid
N_LAYERS   = 16
NOQ_RAW    = 80416        # untrimmed exchanges
RAW_NOQ_BLOCKS = (27056, 27200, 26160)   # untrimmed exchanges per direction (derived from the mapping;
                                         # 27,904 - 26,160 = 1,744 = one layer of columns)

# ---- time -------------------------------------------------------------------------------------
T0          = dt.datetime(2022, 9, 9)   # reference time in the .inp
SIM_START_D = 204                       # 2023-04-01, days after T0
SIM_STOP_D  = 517                       # 2024-02-08
LATITUDE    = 43.6                      # as in the .inp

# ---- monitoring stations: trimmed 1-based segment numbers from the .inp, top to bottom ----------
STATIONS = {
    "CAB": [1114, 2456, 3772, 5050, 6283, 7486, 8626, 9693, 10630],
    "BIS": [1635, 2974, 4283, 5543, 6766, 7964, 9078, 10121, 10973],
}
STATION_TOP_LAYER = 8                   # .his location names are e.g. 'CAB (8)' ... 'CAB (16)'

# ---- forcing (.bin) ---------------------------------------------------------------------------
FORCING_NOFPV = {"wind": "waq-wind-noFPV_remapped_upper.bin",
                 "rad":  "waq-sw_radiation_flux-noFPV_remapped_upper.bin"}

# ---- scenarios: untrimmed FLOW prefix, forcing files, intended full-cover factors ----------------
SCENARIOS = {
    "02": dict(raw=RAW_DIR / "com-02towaq_fpv41_nogaps",
               wind="waq-wind-02towaq_fpv41_nogaps_remapped_upper.bin",
               rad="waq-daily-sw_radiation_flux-02towaq_fpv41_nogaps_remapped_upper.bin",
               wind_factor=0.65, rad_factor=0.65),
    "03": dict(raw=RAW_DIR / "com-03towaq_fpv41_10x10gaps",
               wind="waq-wind-03towaq_fpv41_10x10gaps_remapped_upper.bin",
               rad="waq-daily-sw_radiation_flux-03towaq_fpv41_10x10gaps_remapped_upper.bin",
               wind_factor=0.65, rad_factor=0.65),
    "04": dict(raw=RAW_DIR / "com-04towaq_fpv41_donutgap",
               wind="waq-wind-04towaq_fpv41_donutgap_remapped_upper.bin",
               rad="waq-daily-sw_radiation_flux-04towaq_fpv41_donutgap_remapped_upper.bin",
               wind_factor=0.65, rad_factor=0.65),
}
# NOTE: the SR runs (05/06/07) have FLOW output of only 8,785 records (~183 days) - shorter than the
# simulation. Check their period before adding them here.

# ---- run control ------------------------------------------------------------------------------
STEP = 10                  # long checks read every STEP-th record (1 = full check)
REBUILD_SCENARIOS = []     # e.g. ["02"] - Part D only rebuilds the scenarios listed here
REBUILD_RADIATION = []     # e.g. ["03", "04"] - Part D2 only writes new radiation .bin for these

assert "<MODEL_DIR>" not in str(MODEL_DIR), "Set MODEL_DIR first"
WORK_DIR.mkdir(exist_ok=True)
print("configuration OK")

## 2. Helper functions

Run this cell once. Nothing here reads data yet.

In [ ]:
import os, re, struct, shutil, warnings
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import scipy.io as spio

RESULTS = {}   # name -> (ok, detail); summarised in Section H

def record(name, ok, detail=""):
    RESULTS[name] = (bool(ok), detail)
    print(f"[{'PASS' if ok else 'FAIL'}] {name}" + (f" - {detail}" if detail else ""))

# ---- binary record files ----------------------------------------------------------------------
def rec_bytes(n):
    return (n + 1) * 4

def n_records(path, n):
    """Number of complete records and leftover bytes."""
    return divmod(os.path.getsize(path), rec_bytes(n))

def read_rec(path, n, i):
    with open(path, "rb") as f:
        f.seek(i * rec_bytes(n))
        t = struct.unpack("<i", f.read(4))[0]
        return t, np.frombuffer(f.read(n * 4), "<f4").copy()

def iter_records(path, n, step=1, start=0, stop=None):
    nrec, _ = n_records(path, n)
    stop = nrec if stop is None else min(stop, nrec)
    with open(path, "rb") as f:
        for i in range(start, stop, step):
            f.seek(i * rec_bytes(n))
            t = struct.unpack("<i", f.read(4))[0]
            yield i, t, np.frombuffer(f.read(n * 4), "<f4")

def open_memmap(path, n):
    """Memory-map a segment/exchange record file as a structured array (t, v[n])."""
    dtype = np.dtype([("t", "<i4"), ("v", "<f4", (n,))])
    nrec, rem = n_records(path, n)
    assert rem == 0, f"{path}: not a whole number of {n}-value records"
    return np.memmap(path, dtype=dtype, mode="r", shape=(nrec,))

def progress(i, n, every=2000):
    if (i + 1) % every == 0 or i + 1 == n:
        print(f"    {i+1:,}/{n:,}", end="\r")

# ---- pointer / length / attribute / parameter files ---------------------------------------------
def read_poi(path):
    a = np.fromfile(path, "<i4")
    assert a.size % 4 == 0, f"{path}: size not a multiple of 4 int32"
    return a.reshape(-1, 4)

def read_len(path):
    """Header int + all 'from' lengths + all 'to' lengths -> (header, array (noq, 2))."""
    a = np.fromfile(path, "<f4")
    hdr = int(a[:1].view("<i4")[0])
    v = a[1:]
    assert v.size % 2 == 0
    return hdr, v.reshape(2, -1).T

def read_atr(path, nseg):
    """Parse a DELWAQ attribute file with complete blocks -> list of int arrays (one per block)."""
    t = []
    with open(path, errors="replace") as fh:
        for line in fh:
            t += [int(x) for x in re.findall(r"-?\d+", line.split(";")[0])]
    p = 0; nblocks = t[p]; p += 1; blocks = []
    for _ in range(nblocks):
        nattr = t[p]; p += 1 + nattr
        opt_file, opt_default = t[p], t[p + 1]; p += 2
        assert opt_file == 1 and opt_default == 1, f"unexpected block options {opt_file}, {opt_default}"
        blocks.append(np.array(t[p:p + nseg])); p += nseg
    return blocks, t[p:]

def read_par(path, nseg, npar=2):
    """Binary parameter file: header + values grouped per parameter -> array (npar, nseg)."""
    a = np.fromfile(path, "<f4")
    assert a.size == 1 + npar * nseg, f"{path}: {a.size - 1} values, expected {npar * nseg}"
    return a[1:].reshape(npar, nseg)

# ---- history file -----------------------------------------------------------------------------
def read_his(path):
    with open(path, "rb") as f:
        f.read(160)
        nvar, nloc = struct.unpack("<2i", f.read(8))
        names = [f.read(20).decode(errors="replace").strip() for _ in range(nvar)]
        locs = []
        for _ in range(nloc):
            f.read(4); locs.append(f.read(20).decode(errors="replace").strip())
        data = np.fromfile(f, dtype=np.dtype([("t", "<i4"), ("v", "<f4", (nloc, nvar))]))
    return names, locs, data

# ---- grid geometry ----------------------------------------------------------------------------
def load_segnum(path):
    mat = spio.loadmat(path)
    return mat["data"][0, 0]["Val"]

def grid_positions(segnum, n_seg_in):
    """(m, n, layer) of every untrimmed segment (0-based layer, 0 = top)."""
    valid = ~np.isnan(segnum)
    mm, nn, kk = np.nonzero(valid)
    ids = segnum[valid].astype(int) - 1
    pos = np.full((n_seg_in, 3), -1)
    pos[ids] = np.c_[mm, nn, kk]
    return pos

def array_to_grid(x, new_to_old, segnum, n_seg_in):
    full = np.full(n_seg_in, np.nan); full[new_to_old] = x
    out = np.full(segnum.shape, np.nan); valid = ~np.isnan(segnum)
    out[valid] = full[segnum[valid].astype(int) - 1]
    return out

# ---- .inp parsing -----------------------------------------------------------------------------
def parse_inp(path):
    txt = Path(path).read_text(errors="replace").splitlines()
    info = {"noq": {}, "binary_files": [], "constants": [], "refday": None, "nseg": None}
    in_const = False
    for line in txt:
        l = line.strip()
        m = re.match(r"^(\d+)\s*;\s*number of segments", l, re.I)
        if m: info["nseg"] = int(m.group(1))
        m = re.match(r"^(\d+)\s*;\s*exchanges in direction (\d)", l, re.I)
        if m: info["noq"][int(m.group(2))] = int(m.group(1))
        m = re.match(r"^BINARY_FILE\s+'([^']+)'", l, re.I)
        if m: info["binary_files"].append(m.group(1))
        m = re.match(r"^'([^']+\.(?:vol|poi|are|flo|len|atr))'", l, re.I)
        if m: info["binary_files"].append(m.group(1))
        m = re.match(r"^INCLUDE\s+'([^']+)'", l, re.I)
        if m: info["binary_files"].append(m.group(1))
        m = re.match(r"^([-+0-9.eE]+)\s*;\s*RefDay", l)
        if m: info["refday"] = float(m.group(1))
        if l.upper() == "CONSTANTS": in_const = True; continue
        if l.upper() == "DATA": in_const = False; continue
        if in_const:
            m = re.match(r"^'([^']+)'", l)
            if m: info["constants"].append(m.group(1))
    low = Counter(c.lower() for c in info["constants"])
    info["duplicate_constants"] = sorted({c for c in info["constants"] if low[c.lower()] > 1})
    return info

# ---- trimming ---------------------------------------------------------------------------------
def trim_file(src, dst, n_in, idx, zero_mask=None):
    """Write every complete record of src keeping columns idx; optionally set some output columns to 0.
    Returns (records written, largest |value| that was set to 0)."""
    nrec, rem = n_records(src, n_in)
    max_zeroed = 0.0
    with open(src, "rb") as fi, open(dst, "wb") as fo:
        for i in range(nrec):
            fo.write(fi.read(4))
            v = np.frombuffer(fi.read(n_in * 4), "<f4")[idx].astype("<f4")
            if zero_mask is not None and zero_mask.any():
                max_zeroed = max(max_zeroed, float(np.abs(v[zero_mask]).max()))
                v[zero_mask] = 0.0
            fo.write(v.tobytes())
            progress(i, nrec)
    assert os.path.getsize(dst) == nrec * rec_bytes(len(idx)), "output size mismatch"
    print(f"\n  wrote {Path(dst).name}: {nrec:,} records (dropped {rem} leftover bytes of the source)")
    return nrec, max_zeroed

def compare_trimmed(raw_path, clean_path, n_in, idx, step=1, zero_mask=None):
    """Every step-th record of clean == raw[idx] (with zero_mask columns expected 0)? Returns ok, details."""
    n_r, rem_r = n_records(raw_path, n_in)
    n_c, rem_c = n_records(clean_path, len(idx))
    n = min(n_r, n_c)
    bad_t = bad_v = checked = 0; first_bad = None; max_diff = 0.0; bad_recs = []
    with open(raw_path, "rb") as fr, open(clean_path, "rb") as fc:
        for i in range(0, n, step):
            fr.seek(i * rec_bytes(n_in)); fc.seek(i * rec_bytes(len(idx)))
            tr, vr = fr.read(4), np.frombuffer(fr.read(n_in * 4), "<f4")
            tc, vc = fc.read(4), np.frombuffer(fc.read(len(idx) * 4), "<f4")
            exp = vr[idx].copy()
            if zero_mask is not None:
                exp[zero_mask] = 0.0
            bad_t += tr != tc; checked += 1
            if not np.array_equal(exp, vc):
                bad_v += 1; bad_recs.append(i)
                first_bad = i if first_bad is None else first_bad
                max_diff = max(max_diff, float(np.nanmax(np.abs(exp - vc))))
            if checked % 200 == 0:
                print(f"    {i+1:,}/{n:,}", end="\r")
    ok = (n_r == n_c) and rem_c == 0 and bad_t == 0 and bad_v == 0
    detail = (f"raw {n_r:,} rec (+{rem_r} B) | clean {n_c:,} rec (+{rem_c} B) | checked {checked:,} (step {step}) | "
              f"time mismatches {bad_t} | value mismatches {bad_v} (first at {first_bad}, max|diff| {max_diff:.3e})")
    return ok, detail, bad_recs

# ---- water balance ----------------------------------------------------------------------------
def water_balance(vol_path, flo_path, poi, nseg, n_steps=96, start=2000, min_vol=1.0):
    """|dV - net flow*dt| / V for wet cells over n_steps consecutive steps -> (median, p99, max)."""
    noq = len(poi)
    a, b = poi[:, 0] - 1, poi[:, 1] - 1
    ia, ib = a >= 0, b >= 0
    nv, _ = n_records(vol_path, nseg); nf, _ = n_records(flo_path, noq)
    start = min(start, max(0, min(nv - 1, nf) - n_steps))
    n_steps = min(n_steps, min(nv - 1, nf) - start)
    t0, v0 = read_rec(vol_path, nseg, start)
    errs = []
    for k in range(n_steps):
        _, q = read_rec(flo_path, noq, start + k)
        t1, v1 = read_rec(vol_path, nseg, start + k + 1)
        net = np.zeros(nseg)
        np.add.at(net, a[ia], -q[ia].astype(float) * (t1 - t0))
        np.add.at(net, b[ib], q[ib].astype(float) * (t1 - t0))
        wet = (v0 > min_vol) & (v1 > min_vol)
        errs.append(np.abs((v1 - v0) - net)[wet] / v1[wet])
        t0, v0 = t1, v1
    e = np.concatenate(errs)
    return float(np.median(e)), float(np.percentile(e, 99)), float(e.max())

# ---- astronomy (approximate; only used for timing checks) --------------------------------------
def declination(doy):
    return np.deg2rad(23.44) * np.sin(2 * np.pi * (284 + np.asarray(doy)) / 365)   # Cooper (1969) approximation

def daylength_hours(dates, lat_deg):
    doy = np.array([d.timetuple().tm_yday for d in dates])
    x = -np.tan(np.deg2rad(lat_deg)) * np.tan(declination(doy))
    return 24 / np.pi * np.arccos(np.clip(x, -1, 1))

def toa_daily_mean(dates, lat_deg, S0=1361.0):
    """Approximate daily-mean top-of-atmosphere shortwave (W/m2)."""
    doy = np.array([d.timetuple().tm_yday for d in dates])
    phi, dec = np.deg2rad(lat_deg), declination(doy)
    ecc = 1 + 0.033 * np.cos(2 * np.pi * doy / 365)
    h0 = np.arccos(np.clip(-np.tan(phi) * np.tan(dec), -1, 1))
    return S0 / np.pi * ecc * (h0 * np.sin(phi) * np.sin(dec) + np.cos(phi) * np.cos(dec) * np.sin(h0))

print("helpers loaded")

## Part A — Segment mapping (`new_to_old`)

`new_to_old[i]` is the untrimmed (0-based) segment of trimmed segment `i`. It is the list of segments with volume > 0 at
**any** time in the **baseline** FLOW `.vol`, in original order. The **same** mapping must be used for the baseline and every
scenario, so that segment *i* is the same cell in every run (and matches the baseline `.poi`, `.atr`, `.par`, forcing).

Building it reads the whole untrimmed baseline `.vol` (≈1.7 GB). Once saved, it is loaded instead.

*Why not use each scenario's own wet cells?* That is what shifted scenario 02 by one segment: FLOW 02 is wet in one extra cell
(original index 1568), so "first 10,999 of its own mask" misaligned every later segment.

In [ ]:
def ever_wet(vol_path, nseg):
    nrec, rem = n_records(vol_path, nseg)
    act, vmax = np.zeros(nseg, bool), np.zeros(nseg)
    for i, t, v in iter_records(vol_path, nseg):
        act |= v > 0
        np.maximum(vmax, v, out=vmax)
        progress(i, nrec)
    print()
    return act, vmax

MAP_FILE = WORK_DIR / "new_to_old.npy"
if MAP_FILE.exists():
    new_to_old = np.load(MAP_FILE)
    print(f"loaded {MAP_FILE}")
else:
    print("building mapping from the untrimmed baseline .vol ...")
    act_base, _ = ever_wet(f"{RAW_BASE}.vol", N_SEG_IN)
    new_to_old = np.flatnonzero(act_base)
    np.save(MAP_FILE, new_to_old)
    print(f"saved {MAP_FILE}")

record("A: mapping has N_SEG segments", len(new_to_old) == N_SEG, f"{len(new_to_old):,} segments")
record("A: mapping strictly increasing (original order)", bool(np.all(np.diff(new_to_old) > 0)))

old_to_new1 = np.zeros(N_SEG_IN + 1, np.int64)            # untrimmed 1-based id -> trimmed 1-based id (0 = dropped)
old_to_new1[new_to_old + 1] = np.arange(1, N_SEG + 1)

segnum  = load_segnum(SEGNUM_MAT)
pos_all = grid_positions(segnum, N_SEG_IN)                 # (m, n, layer) per untrimmed segment
pos     = pos_all[new_to_old]                              # (m, n, layer) per trimmed segment
record("A: every segment has a grid position", bool((pos_all[:, 0] >= 0).all()))

# columns of the trimmed grid: top (surface) segment and the one below it, per column
cols = {}
for s in range(N_SEG):
    cols.setdefault(tuple(pos[s, :2]), []).append((pos[s, 2], s))
top_seg = np.array([sorted(c)[0][1] for c in cols.values()])
second  = np.array([sorted(c)[1][1] if len(c) > 1 else -1 for c in cols.values()])
print(f"{len(cols):,} columns in the trimmed grid")

### A2 (optional) — cells that are wet in a scenario but not in the baseline

These cells are dropped when the baseline mapping is used. The check reports how many there are and how much water they
can hold (found earlier: cell 1568 in 02/03/04 and 10562 in 04, at most ≈2 m³). Each scenario reads a full untrimmed `.vol`.

In [ ]:
RUN_A2 = False     # set True to scan the scenarios listed in SCENARIOS
if RUN_A2:
    base_mask = np.zeros(N_SEG_IN, bool); base_mask[new_to_old] = True
    for scen, cfg in SCENARIOS.items():
        print(f"scenario {scen}: scanning {cfg['raw'].name}.vol")
        act, vmax = ever_wet(f"{cfg['raw']}.vol", N_SEG_IN)
        extra = np.flatnonzero(act & ~base_mask)
        never = np.flatnonzero(base_mask & ~act)
        print(f"  ever-wet {act.sum():,} | extra vs baseline: {extra.size} at {extra[:10].tolist()} "
              f"(max volume {vmax[extra].max() if extra.size else 0:.2f} m3) | baseline cells never wet here: {never.size}")
        record(f"A2: scenario {scen} extra wet cells hold < 10 m3", (vmax[extra].max() if extra.size else 0) < 10,
               f"{extra.size} extra cells")

## Part B — Verify the baseline trimmed files

Checks the files in `BL_DIR` (the ones the baseline run read) against the untrimmed FLOW output.

### B1 — Dimensions declared in the `.inp`

In [ ]:
inp = parse_inp(BASE_INP)
noq_inp = tuple(inp["noq"][k] for k in (1, 2, 3))
cln_poi = read_poi(BL_DIR / f"{CLEAN}.poi")
NOQ = len(cln_poi)
print(f".inp: {inp['nseg']} segments | exchanges per direction {noq_inp} | RefDay {inp['refday']}")
print(f"files referenced: {inp['binary_files']}")
record("B1: .inp segments == N_SEG", inp["nseg"] == N_SEG)
record("B1: .inp exchanges == trimmed .poi rows", sum(noq_inp) == NOQ, f"{sum(noq_inp):,} vs {NOQ:,}")
per_layer = inp["nseg"] - noq_inp[2]
print(f"DELWAQ layer estimate: segments - vertical exchanges = {per_layer:,} "
      f"({'consistent' if per_layer > 0 and inp['nseg'] % per_layer == 0 else 'inconsistent -> DELWAQ assumes ONE layer'})")
if inp["duplicate_constants"]:
    print(f"WARNING duplicate constants (case-insensitive): {inp['duplicate_constants']}")

### B2 — Segment files `.vol`, `.tem`, `.vdf`

Every `STEP`-th record of the trimmed file must equal the untrimmed record in `new_to_old` order, with identical time stamps.

In [ ]:
for ext in ("vol", "tem", "vdf"):
    ok, detail, _ = compare_trimmed(f"{RAW_BASE}.{ext}", BL_DIR / f"{CLEAN}.{ext}", N_SEG_IN, new_to_old, step=STEP)
    record(f"B2: baseline .{ext}", ok, detail)

### B3 — Attribute file `.atr`

* Block 1 (active) must be 1 for all kept segments.
* Block 2 (position): the trimmed file was **rebuilt** so that the first ever-wet segment of each column is the surface
  (code 1, or 0 for single-segment columns). Here we check every column has exactly one surface-coded segment and it is the top one.

In [ ]:
atr_raw, _ = read_atr(f"{RAW_BASE}.atr", N_SEG_IN)
atr_cln, rest = read_atr(BL_DIR / f"{CLEAN}.atr", N_SEG)
record("B3: .atr has 2 blocks", len(atr_cln) == 2, f"leftover tokens {rest}")
record("B3: all kept segments active", bool((atr_cln[0] == 1).all()))
surf_codes = atr_cln[1][top_seg]
n_surf_per_col = [sum(atr_cln[1][s] in (0, 1) for _, s in c) for c in cols.values()]
record("B3: top segment of every column is surface-coded (0/1)", bool(np.isin(surf_codes, [0, 1]).all()),
       f"{len(cols):,} columns")
record("B3: exactly one surface-coded segment per column", all(n == 1 for n in n_surf_per_col))
print("position-code counts:", dict(sorted(Counter(atr_cln[1].tolist()).items())))

### B4 — `SURF` in the `.par` vs the untrimmed `.srf`

In [ ]:
par = read_par(BASE_PAR, N_SEG)
srf = np.fromfile(f"{RAW_BASE}.srf", "<f4")
if srf.size == N_SEG_IN + 1:
    record("B4: SURF == untrimmed .srf[new_to_old]", np.allclose(par[0], srf[1:][new_to_old], rtol=1e-6))
else:
    print(f".srf has {srf.size} values - layout differs from the assumed header + {N_SEG_IN}; check skipped")
print(f"SURF range {par[0].min():.3g}-{par[0].max():.3g} | bottomdept range {par[1].min():.3g}-{par[1].max():.3g}")

### B5 — Monitoring stations

Each station's segments must lie in one column, in consecutive layers.

In [ ]:
for st, segs in STATIONS.items():
    p = pos[np.array(segs) - 1]
    one_col = len({tuple(x) for x in p[:, :2]}) == 1
    consecutive = bool(np.all(np.diff(p[:, 2]) == 1))
    record(f"B5: station {st} in one column, consecutive layers", one_col and consecutive,
           f"column {tuple(p[0, :2])}, layer indices {p[:, 2].tolist()}")

### B6 — Pointers connect neighbouring cells

For every exchange with a segment at both ends, the two cells must be grid neighbours (distance 1 in m, n or layer).
This shows `.poi` and `.vol` use the same numbering.

In [ ]:
both = (cln_poi[:, 0] > 0) & (cln_poi[:, 1] > 0)
d = np.abs(pos[cln_poi[both, 0] - 1] - pos[cln_poi[both, 1] - 1]).sum(axis=1)
record("B6: all exchanges link neighbouring cells", bool(np.all(d == 1)), f"{both.sum():,} exchanges, {np.mean(d == 1):.4%}")

### B7 — Exchange mapping (`exch_map`) and `.poi`, `.len`, `.flo`, `.are`

`exch_map[i]` is the untrimmed row of trimmed exchange `i`. The trimming kept rows by a rule that is not simply
"ever-wet ends" (it includes ~14,000 rows whose ends were both removed), so the mapping is **reconstructed** rather than
re-derived:

1. Renumber the untrimmed pointers with `new_to_old` (removed segments → 0).
2. Within each direction block, walk both files in order and match rows with identical renumbered pointers, both lengths and the area at t = 0.
3. Identical-looking rows (e.g. `0 0 0 0` with equal lengths and areas) can be swapped; any `.flo` record that then differs is
   added to the matching key, and the match is repeated (up to 3 times).

The mapping is saved only if `.poi`, `.len`, `.flo` and `.are` all pass.

In [ ]:
raw_poi = read_poi(f"{RAW_BASE}.poi")
assert len(raw_poi) == NOQ_RAW and raw_poi.min() >= 0, "unexpected untrimmed .poi (size or open boundaries)"
conv = old_to_new1[raw_poi]
_, L_raw = read_len(f"{RAW_BASE}.len")
_, L_cln = read_len(BL_DIR / f"{CLEAN}.len")
_, a_r0 = read_rec(f"{RAW_BASE}.are", NOQ_RAW, 0)
_, a_c0 = read_rec(BL_DIR / f"{CLEAN}.are", NOQ, 0)

RAW_EDGES = np.cumsum((0,) + RAW_NOQ_BLOCKS)
CLN_EDGES = np.cumsum((0,) + noq_inp)

def ordered_match(key_r, key_c):
    m = np.full(len(key_c), -1)
    for b in range(3):
        j = RAW_EDGES[b]
        for i in range(CLN_EDGES[b], CLN_EDGES[b + 1]):
            while j < RAW_EDGES[b + 1] and key_r[j] != key_c[i]:
                j += 1
            if j == RAW_EDGES[b + 1]:
                break
            m[i] = j; j += 1
    return m

key_r = [tuple(conv[j]) + (float(L_raw[j, 0]), float(L_raw[j, 1]), float(a_r0[j])) for j in range(NOQ_RAW)]
key_c = [tuple(cln_poi[i]) + (float(L_cln[i, 0]), float(L_cln[i, 1]), float(a_c0[i])) for i in range(NOQ)]
exch_map = ordered_match(key_r, key_c)
print(f"matched {(exch_map >= 0).sum():,} of {NOQ:,}")

FLO_R, FLO_C = f"{RAW_BASE}.flo", BL_DIR / f"{CLEAN}.flo"
for it in range(3):
    if (exch_map < 0).any():
        break
    ok_flo, detail_flo, bad = compare_trimmed(FLO_R, FLO_C, NOQ_RAW, exch_map, step=STEP)
    if ok_flo:
        break
    recs = bad[:20]
    print(f"refinement {it+1}: adding flows of records {recs} to the key")
    sig_r = np.stack([read_rec(FLO_R, NOQ_RAW, r)[1] for r in recs], axis=1)
    sig_c = np.stack([read_rec(FLO_C, NOQ, r)[1] for r in recs], axis=1)
    key_r = [key_r[j] + tuple(map(float, sig_r[j])) for j in range(NOQ_RAW)]
    key_c = [key_c[i] + tuple(map(float, sig_c[i])) for i in range(NOQ)]
    exch_map = ordered_match(key_r, key_c)

matched = bool((exch_map >= 0).all())
record("B7: every trimmed exchange matched in order", matched, f"{(exch_map >= 0).sum():,} of {NOQ:,}")
if matched:
    record("B7: .poi == renumbered untrimmed .poi", np.array_equal(conv[exch_map], cln_poi))
    record("B7: .len == untrimmed .len", np.array_equal(L_raw[exch_map], L_cln))
    ok_flo, detail_flo, _ = compare_trimmed(FLO_R, FLO_C, NOQ_RAW, exch_map, step=STEP)
    record("B7: .flo", ok_flo, detail_flo)
    ok_are, detail_are, _ = compare_trimmed(f"{RAW_BASE}.are", BL_DIR / f"{CLEAN}.are", NOQ_RAW, exch_map, step=STEP)
    record("B7: .are", ok_are, detail_are)
    for b in range(3):
        sub = exch_map[CLN_EDGES[b]:CLN_EDGES[b + 1]]
        record(f"B7: direction {b+1} maps inside untrimmed block {b+1}",
               bool(((sub >= RAW_EDGES[b]) & (sub < RAW_EDGES[b + 1])).all()))
    keep = all(RESULTS[k][0] for k in ("B7: .poi == renumbered untrimmed .poi", "B7: .len == untrimmed .len",
                                       "B7: .flo", "B7: .are"))
    if keep:
        np.save(WORK_DIR / "exch_map.npy", exch_map)
        print(f"saved {WORK_DIR / 'exch_map.npy'}")
    else:
        print("exch_map NOT saved")

zero_ptr = (cln_poi[:, 0] == 0) | (cln_poi[:, 1] == 0)
print(f"exchanges with a 0 pointer: {zero_ptr.sum():,} (flows on these are 0 in the baseline)")

### B8 — Water balance of the baseline files

For wet cells, the volume change between two records must equal the net flow × Δt. Small values (median ≈ 1e-9) mean volumes
and flows are consistent. Isolated larger values occur in small or wetting/drying cells.

In [ ]:
wb_base = water_balance(BL_DIR / f"{CLEAN}.vol", BL_DIR / f"{CLEAN}.flo", cln_poi, N_SEG)
print(f"baseline |dV - net flow| / V: median {wb_base[0]:.2e} | 99th pct {wb_base[1]:.2e} | max {wb_base[2]:.2e}")
record("B8: baseline water balance median < 1e-6", wb_base[0] < 1e-6)

## Part C — Forcing checks (radiation and wind `.bin`)

For each scenario, against the no-FPV files:

* time axis: regular steps, starting at T0, covering the simulation (days 204–517);
* open water identical to no-FPV; changed cells identical for wind and radiation (same footprint);
* factor in fully covered cells equals the intended factor (0.65), and partly covered cells follow the
  area-weighted rule `factor = 1 − (1 − F) × covered_fraction`, with the **same** covered fraction for wind and radiation;
* no-FPV radiation never exceeds the top-of-atmosphere daily mean (a timing/units sanity check).

Radiation files are read completely (554 daily records); wind (10-minute records, ≈3.5 GB) is sampled at 24 times in the simulation.

In [ ]:
def covered_fraction(ratio, factor):
    return (1 - ratio) / (1 - factor)

def check_forcing(scen, cfg, n_snap=24, tol=1e-5):
    warnings.filterwarnings("ignore", message="All-NaN slice")   # cells without forcing (deeper layers)
    out = {}
    for kind in ("rad", "wind"):
        m_ref = open_memmap(FORCING_DIR / FORCING_NOFPV[kind], N_SEG)
        m_scn = open_memmap(FORCING_DIR / cfg[kind], N_SEG)
        t = np.asarray(m_scn["t"])
        steps = np.unique(np.diff(t))
        covers = t[0] <= SIM_START_D * 86400 and t[-1] >= SIM_STOP_D * 86400
        record(f"C {scen} {kind}: time axis regular, from T0, covers simulation",
               len(steps) == 1 and t[0] == 0 and covers and np.array_equal(t, np.asarray(m_ref["t"])),
               f"{len(t):,} records, step {steps.tolist()} s, day {t[0]/86400:.2f}-{t[-1]/86400:.2f}")
        if kind == "rad":
            idx = np.arange(len(t))
        else:
            i0, i1 = np.searchsorted(t, SIM_START_D * 86400), np.searchsorted(t, SIM_STOP_D * 86400)
            idx = np.linspace(i0, min(i1, len(t) - 1), n_snap).astype(int)
        R = np.asarray(m_ref["v"][idx], dtype=float)
        S = np.asarray(m_scn["v"][idx], dtype=float)
        changed = np.any(~np.isclose(S, R, rtol=1e-6, atol=1e-6), axis=0)
        ratio = np.divide(S, R, out=np.full_like(S, np.nan), where=R > 0.1)
        med = np.nanmedian(np.where(changed, ratio, np.nan), axis=0) if changed.any() else np.full(N_SEG, np.nan)
        spread = np.nanmax(ratio[:, changed], axis=0) - np.nanmin(ratio[:, changed], axis=0) if changed.any() else np.array([0.0])
        F = cfg[f"{kind}_factor"]
        full_cover = np.nanmin(med[changed]) if changed.any() else np.nan
        record(f"C {scen} {kind}: full-cover factor == {F}", abs(full_cover - F) < tol, f"found {full_cover:.4f}")
        record(f"C {scen} {kind}: factor constant in time per cell", np.nanmax(spread) < tol, f"max spread {np.nanmax(spread):.1e}")
        out[kind] = dict(changed=changed, frac=np.where(changed, covered_fraction(med, F), 0.0))
    same_fp = np.array_equal(out["rad"]["changed"], out["wind"]["changed"])
    record(f"C {scen}: wind and radiation footprints identical", same_fp, f"{out['rad']['changed'].sum():,} cells")
    if same_fp:
        dmax = np.nanmax(np.abs(out["rad"]["frac"] - out["wind"]["frac"]))
        record(f"C {scen}: same covered fraction for wind and radiation", dmax < 1e-4, f"max diff {dmax:.1e}")
    print(f"  covered fractions present: {np.unique(np.round(out['wind']['frac'][out['wind']['changed']], 3)).tolist()}")
    return out

FORCING = {scen: check_forcing(scen, cfg) for scen, cfg in SCENARIOS.items()}

# no-FPV radiation vs top-of-atmosphere (open-water segment with the largest mean radiation)
m_rad = open_memmap(FORCING_DIR / FORCING_NOFPV["rad"], N_SEG)
rad = np.asarray(m_rad["v"], dtype=float)
seg_open = int(np.argmax(rad.mean(axis=0)))
dates = np.array([T0 + dt.timedelta(seconds=int(s)) for s in m_rad["t"]])
toa = toa_daily_mean(dates, LATITUDE)
exceed = rad[:, seg_open] > toa
record("C noFPV radiation never above TOA daily mean", exceed.sum() == 0, f"{exceed.sum()} of {len(dates)} days")
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(dates, rad[:, seg_open], lw=0.8, label=f"noFPV radiation, segment {seg_open}")
ax.plot(dates, toa, "k--", lw=1, label=f"TOA daily mean {LATITUDE}°N (approx.)")
ax.axvspan(T0 + dt.timedelta(days=SIM_START_D), T0 + dt.timedelta(days=SIM_STOP_D), color="0.9", label="simulation")
ax.set_ylabel("W/m²"); ax.legend(fontsize=8); fig.tight_layout()

## Part D — Rebuild scenario hydrodynamics

For each scenario in `REBUILD_SCENARIOS`, writes `MODEL_DIR/XXscen_fixed/` containing:

| File | Source |
|---|---|
| `.vol`, `.tem`, `.vdf` | the scenario's **own** untrimmed FLOW files, columns `new_to_old` (baseline mapping) |
| `.flo`, `.are` | the scenario's **own** untrimmed FLOW files, columns `exch_map`; flows on exchanges with a 0 pointer set to 0 (as in the baseline) |
| `.poi`, `.len`, `.atr`, `.par` | copied from the baseline (geometry, identical for all runs) |

Then it verifies the output against the untrimmed scenario files, checks the files come from the scenario (not the baseline),
and compares the water balance with the baseline. The largest flow set to 0 also measures what dropping the scenario's extra wet
cells costs. Forcing `.bin` files and the `.inp` are **not** written here (see D2 and Part G).

In [ ]:
cln_poi  = read_poi(BL_DIR / f"{CLEAN}.poi")              # baseline pointers (also defined in Part B)
NOQ      = len(cln_poi)
zero_ptr = (cln_poi[:, 0] == 0) | (cln_poi[:, 1] == 0)
exch_map = np.load(WORK_DIR / "exch_map.npy")
assert len(exch_map) == NOQ

def rebuild_scenario(scen, cfg):
    out_dir = MODEL_DIR / f"{scen}scen_fixed"
    out_dir.mkdir(exist_ok=True)
    print(f"\n=== scenario {scen} -> {out_dir}")
    for ext in ("vol", "tem", "vdf"):
        trim_file(f"{cfg['raw']}.{ext}", out_dir / f"{CLEAN}.{ext}", N_SEG_IN, new_to_old)
    _, max_zeroed = trim_file(f"{cfg['raw']}.flo", out_dir / f"{CLEAN}.flo", NOQ_RAW, exch_map, zero_mask=zero_ptr)
    print(f"  largest |flow| set to 0 on 0-pointer exchanges: {max_zeroed:.3e} m3/s")
    trim_file(f"{cfg['raw']}.are", out_dir / f"{CLEAN}.are", NOQ_RAW, exch_map)
    for f in (f"{CLEAN}.poi", f"{CLEAN}.len", f"{CLEAN}.atr", BASE_PAR.name):
        shutil.copy2(BL_DIR / f, out_dir / f)
    print("  copied .poi, .len, .atr, .par from the baseline")

    for ext, n_in, idx, zm in (("vol", N_SEG_IN, new_to_old, None), ("tem", N_SEG_IN, new_to_old, None),
                               ("vdf", N_SEG_IN, new_to_old, None), ("flo", NOQ_RAW, exch_map, zero_ptr),
                               ("are", NOQ_RAW, exch_map, None)):
        ok, detail, _ = compare_trimmed(f"{cfg['raw']}.{ext}", out_dir / f"{CLEAN}.{ext}", n_in, idx, step=STEP, zero_mask=zm)
        record(f"D {scen}: .{ext} == own FLOW output", ok, detail)
    r_mid = min(2000, n_records(BL_DIR / f"{CLEAN}.vol", N_SEG)[0] - 1)
    _, vb = read_rec(BL_DIR / f"{CLEAN}.vol", N_SEG, r_mid)
    _, vs = read_rec(out_dir / f"{CLEAN}.vol", N_SEG, r_mid)
    record(f"D {scen}: volumes differ from baseline (real FPV effect)", not np.array_equal(vb, vs),
           f"max|dV| {np.max(np.abs(vs - vb)):.3f} m3 at record {r_mid}")
    wb = water_balance(out_dir / f"{CLEAN}.vol", out_dir / f"{CLEAN}.flo", cln_poi, N_SEG)
    print(f"  water balance: median {wb[0]:.2e} | 99th pct {wb[1]:.2e} | max {wb[2]:.2e} "
          f"(baseline {wb_base[0]:.2e} | {wb_base[1]:.2e} | {wb_base[2]:.2e})")
    record(f"D {scen}: water balance median within 10x of baseline", wb[0] <= 10 * max(wb_base[0], 1e-12))
    nv, _ = n_records(out_dir / f"{CLEAN}.vol", N_SEG)
    t_last = read_rec(out_dir / f"{CLEAN}.vol", N_SEG, nv - 1)[0]
    record(f"D {scen}: hydrodynamics cover the simulation", t_last >= SIM_STOP_D * 86400, f"last day {t_last/86400:.2f}")

if "wb_base" not in globals():
    wb_base = water_balance(BL_DIR / f"{CLEAN}.vol", BL_DIR / f"{CLEAN}.flo", cln_poi, N_SEG)
for scen in REBUILD_SCENARIOS:
    rebuild_scenario(scen, SCENARIOS[scen])

### D2 (optional) — Rebuild a radiation `.bin` with the intended factor

Scenarios 03 and 04 were found to use 0.10 instead of 0.65 under the panels. If the original forcing script is not at hand, a
corrected file can be derived from what has been verified:

* the covered fraction of every segment is taken from the scenario's **wind** file (wind factor verified as 0.65, constant in time);
* radiation = no-FPV radiation × `(1 − (1 − F) × covered_fraction)` with `F = rad_factor` from the configuration.

This reproduces the area-weighted rule found in the existing files (scenario 02 matches it exactly). **If you have the original
script that generated the forcing, prefer re-running that with the corrected factor.** The new file gets a new name; re-run Part C on it
(point `SCENARIOS[scen]['rad']` to the new file) before using it.

In [ ]:
def rebuild_radiation(scen, cfg, n_snap=24):
    m_wref = open_memmap(FORCING_DIR / FORCING_NOFPV["wind"], N_SEG)
    m_wscn = open_memmap(FORCING_DIR / cfg["wind"], N_SEG)
    t = np.asarray(m_wscn["t"])
    i0, i1 = np.searchsorted(t, SIM_START_D * 86400), np.searchsorted(t, SIM_STOP_D * 86400)
    idx = np.linspace(i0, min(i1, len(t) - 1), n_snap).astype(int)
    W0 = np.asarray(m_wref["v"][idx], dtype=float); W = np.asarray(m_wscn["v"][idx], dtype=float)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        ratio = np.nanmedian(np.divide(W, W0, out=np.full_like(W, np.nan), where=W0 > 0.1), axis=0)
    frac = np.where(np.isnan(ratio), 0.0, covered_fraction(ratio, cfg["wind_factor"]))
    frac = np.clip(np.round(frac, 6), 0, 1)
    m_rref = open_memmap(FORCING_DIR / FORCING_NOFPV["rad"], N_SEG)
    factor = 1 - (1 - cfg["rad_factor"]) * frac
    out = FORCING_DIR / cfg["rad"].replace(".bin", f"_F{int(round(cfg['rad_factor'] * 100)):03d}_rebuilt.bin")
    assert not out.exists(), f"{out} exists - delete it first if you want to rebuild"
    with open(out, "wb") as fo:
        for rec in m_rref:
            fo.write(np.int32(rec["t"]).tobytes())
            fo.write((rec["v"] * factor).astype("<f4").tobytes())
    print(f"  wrote {out.name} | covered fractions: {np.unique(frac[frac > 0]).round(3).tolist()}")
    return out

for scen in REBUILD_RADIATION:
    new_file = rebuild_radiation(scen, SCENARIOS[scen])
    cfg_new = dict(SCENARIOS[scen], rad=new_file.name)
    check_forcing(f"{scen}-rebuilt", cfg_new)

## Part E — Audit existing clean folders

Identifies which FLOW run a trimmed `.vol` was built from, and whether its segments are in baseline order. For each clean segment
it looks for the untrimmed segment with exactly the same volumes at three records. (Deep, always-full cells have identical volumes
in every run, so `found` ≈ 0.7 for the wrong run and 1.0 for the right one.)

In [ ]:
AUDIT = {   # label: trimmed .vol to audit   - edit as needed
    "02scen (old)":   MODEL_DIR / "02scen" / f"{CLEAN}.vol",
    "02scen_fixed":   MODEL_DIR / "02scen_fixed" / f"{CLEAN}.vol",
}
CANDIDATES = {"baseline": RAW_BASE, **{f"FLOW {s}": c["raw"] for s, c in SCENARIOS.items()}}

def identify_source(clean_vol, raw_prefix, recs=None):
    if recs is None:
        n = min(n_records(clean_vol, N_SEG)[0], n_records(f"{raw_prefix}.vol", N_SEG_IN)[0])
        recs = tuple(int(f * (n - 1)) for f in (0.15, 0.35, 0.55))
    C = np.stack([read_rec(clean_vol, N_SEG, i)[1] for i in recs], 1)
    Rr = np.stack([read_rec(f"{raw_prefix}.vol", N_SEG_IN, i)[1] for i in recs], 1)
    times_equal = all(read_rec(clean_vol, N_SEG, i)[0] == read_rec(f"{raw_prefix}.vol", N_SEG_IN, i)[0] for i in recs)
    lookup = {}
    for j, row in enumerate(map(tuple, Rr)):
        lookup.setdefault(row, []).append(j)
    nonzero = C.any(axis=1)
    hits = [lookup.get(tuple(r), []) for r in C]
    found = np.mean([len(h) > 0 for h, nz in zip(hits, nonzero) if nz])
    unique = np.array([h[0] if len(h) == 1 else -1 for h in hits])
    ok = (unique >= 0) & nonzero
    order = float(np.mean(unique[ok] == new_to_old[ok])) if ok.any() else np.nan
    return dict(times_equal=times_equal, found=round(float(found), 4), unique=int(ok.sum()), baseline_order=round(order, 4))

for lab, p in AUDIT.items():
    if not Path(p).exists():
        print(f"{lab}: not found ({p})"); continue
    print(f"\n{lab}:")
    for src, raw in CANDIDATES.items():
        if Path(f"{raw}.vol").exists():
            print(f"  vs {src:10s}: {identify_source(p, raw)}")

## Part F — Model diagnostics from a finished run

Point `RUN_DIR` to a run folder. Checks:

* **`.lst` warnings**, in particular "the program will now assume one (1) layer";
* **day length**: modelled `DayL` vs astronomical day length; the best-fit lag reveals a wrong `RefDay` (with `RefDay = 204` the model was ≈48 days behind: peak on 8 Aug instead of 21 Jun). Needs a run of at least ~30 days;
* **`Depth`** reported by DELWAQ vs volume / `SURF` at the stations (found 6–79 m below the thermocline instead of 0.92 m);
* **dry surface segment**: how often the surface-coded segment is empty while the one below is wet.

In [ ]:
RUN_DIR = BL_DIR
RUN_NAME = "Torun_WAQ_Apr2023"

lst = RUN_DIR / f"{RUN_NAME}.lst"
if lst.exists():
    for i, line in enumerate(lst.read_text(errors="replace").splitlines(), 1):
        if re.search(r"warning|error|assume one", line, re.I):
            print(f"{i:6d}: {line.strip()[:150]}")

names, locs, his = read_his(RUN_DIR / f"{RUN_NAME}.his")
his_dates = np.array([T0 + dt.timedelta(seconds=int(s)) for s in his["t"]])
print(f"\n.his: {len(names)} variables, {len(locs)} locations, {len(his)} times")

# ---- day length ----
st0 = list(STATIONS)[0]
loc0 = f"{st0} ({STATION_TOP_LAYER})"
if "DayL" in names and loc0 in locs:
    dayl = his["v"][:, locs.index(loc0), names.index("DayL")].astype(float)
    dayl_h = dayl * 24 if np.nanmax(dayl) <= 1.5 else dayl
    astro = daylength_hours(his_dates, LATITUDE)
    # best-fit shift: astronomical day length evaluated L days earlier; L = +48 reproduces RefDay = 204
    days = np.array([(d - T0).total_seconds() / 86400 for d in his_dates])
    lags = np.arange(-120, 121)
    err = [np.mean(np.abs(dayl_h - daylength_hours([T0 + dt.timedelta(days=x - L) for x in days], LATITUDE)))
           for L in lags]
    best = int(lags[int(np.argmin(err))])
    span = days[-1] - days[0]
    record("F: modelled day length matches the calendar (best-fit lag within 3 days)", abs(best) <= 3,
           f"best-fit lag {best:+d} days (model behind calendar if positive), mean |error| {min(err):.2f} h, "
           f"run spans {span:.0f} days" + (" - too short to judge" if span < 30 else ""))
    fig, ax = plt.subplots(figsize=(11, 3.2))
    ax.plot(his_dates, dayl_h, label="DayL (model)"); ax.plot(his_dates, astro, "k--", lw=1, label="astronomical (approx.)")
    ax.set_ylabel("hours"); ax.legend(fontsize=8); fig.tight_layout()

# ---- Depth vs volume / SURF at the stations ----
par_run = read_par(RUN_DIR / BASE_PAR.name, N_SEG)
ih = len(his) // 2
t_mid = int(his["t"][ih])
vol_path = RUN_DIR / f"{CLEAN}.vol"
t_first = read_rec(vol_path, N_SEG, 0)[0]
iv = (t_mid - t_first) // (read_rec(vol_path, N_SEG, 1)[0] - t_first)
tv, vol_mid = read_rec(vol_path, N_SEG, iv)
print(f"\nDepth check at {T0 + dt.timedelta(seconds=t_mid):%Y-%m-%d %H:%M} (vol t={tv}, his t={t_mid})")
bad_depth = 0
for st, segs in STATIONS.items():
    for k, s in enumerate(segs):
        loc = f"{st} ({STATION_TOP_LAYER + k})"
        if loc not in locs or "Depth" not in names:
            continue
        d_his = his["v"][ih, locs.index(loc), names.index("Depth")]
        d_file = vol_mid[s - 1] / par_run[0][s - 1]
        flag = "" if abs(d_his - d_file) < 0.05 * max(d_file, 0.01) + 0.01 else "   <-- differs"
        bad_depth += bool(flag)
        print(f"  {loc:10s} seg {s:6d}: vol/SURF {d_file:7.3f} m | DELWAQ Depth {d_his:8.3f} m{flag}")
record("F: DELWAQ Depth == volume/SURF at all station segments", bad_depth == 0, f"{bad_depth} differ")

# ---- dry surface segment ----
nv, _ = n_records(vol_path, N_SEG)
bad_per_col = np.zeros(len(top_seg), int); steps_bad = 0
for i, t, v in iter_records(vol_path, N_SEG, step=STEP):
    bad = (v[top_seg] <= 1e-6) & (second >= 0) & (v[np.maximum(second, 0)] > 1e-6)
    steps_bad += bool(bad.any()); bad_per_col += bad
    progress(i, nv)
n_checked = len(range(0, nv, STEP))
print(f"\nchecked {n_checked:,} records: steps with a dry surface segment over water: {steps_bad:,} "
      f"({steps_bad/n_checked:.1%}) | columns ever affected: {(bad_per_col > 0).sum():,} of {len(top_seg):,}")
for st, segs in STATIONS.items():
    c = int(np.flatnonzero(top_seg == segs[0] - 1)[0]) if (segs[0] - 1) in top_seg else None
    if c is not None:
        print(f"  {st}: surface segment dry over water in {bad_per_col[c]/n_checked:.1%} of checked records")

## Part G — `.inp` checklist and open issues

**Before re-running the baseline and scenarios** (keep all runs identical except the scenario-specific lines):

| Item | Where | Change |
|---|---|---|
| `RefDay` | constants | `204` → **`252`** (day of year of T0 = 9 Sep 2022). Verify the definition of `RefDay` in the D-WAQ processes manual. Changes `DayL` and therefore BLOOM — may require recalibration. |
| Duplicate switches | constants | Keep **one** entry each of `ACTIVE_TotDepth`, `ACTIVE_VertDisp`, `ACTIVE_DynDepth` (DELWAQ reads names case-insensitively; `ACTIVE_TOTDEPTH` was read as 0 and 1 into the same constant). |
| Map output list | block 9 | Add a 4th list (map) between the history list and `1 ; binary history file on`. Without it the following switches shift by one line (the `.lst` "unexpected end of datagroup" error; NEFIS map switched off; `.map` only holds substances). |
| Hydrodynamic files | blocks 3–4, segment functions | Scenario folder `XXscen_fixed` (all names stay `com-WQexport_clean.*`). |
| Forcing | segment functions | `VWind`, `RadAve`, `RadSurf` → the scenario's verified `.bin` files. |

**Diagnostic run** (short, e.g. to 2023-07-15): add to the history list `Rad`, `RadBot`, `LocalDepth`, `TotalDepth`, `SURF` and the
reaeration flux (exact name from the `found flux` lines of `RearOXY` in the `.lsp`; possibly `dREAROXY` — not verified), and
`Depth`, `LocalDepth`, `TotalDepth`, `Rad`, `RadBot`, `SURF` to the new map list.

**Open model-level issues** (affect the baseline and every scenario equally; not caused by the trimming):

1. DELWAQ warns that segments − vertical exchanges is inconsistent and **assumes one layer**. On the untrimmed grid it would be consistent (27,904 − 26,160 = 1,744). Effects on processes unknown.
2. **`Depth` from `DynDepth`** is far larger than volume/`SURF` in many segments below the thermocline (CAB 13–16, BIS 12–16), although volumes, `SURF` and exchange areas are correct. `Depth` feeds light, BLOOM, settling and reaeration. Cause unknown — inspect the `Input for [DynDepth]` block of the `.lsp` and ask Deltares support.
3. The **surface segment is empty** for long periods (≈50% at CAB), while surface code, `RadSurf` and `VWind` stay on that segment.
4. One process uses **`TotalDepth` = constant 8.6 m** (`.lsp` lines around "Using constant nr 1").
5. An alternative to test: the **full grid** (27,904 segments) with `MULTIGRID ZMODEL NOLAY 16 END_MULTIGRID`, as suggested by the DELWAQ warning, and an `.atr` that puts the surface code on the first wet cell. Not yet tested — verify in the D-WAQ input manual.

## H. Summary of all checks run in this session

In [ ]:
width = max((len(k) for k in RESULTS), default=10)
for name, (ok, detail) in RESULTS.items():
    print(f"{'PASS' if ok else 'FAIL'}  {name:{width}s}  {detail}")
print(f"\n{sum(ok for ok, _ in RESULTS.values())} PASS, {sum(not ok for ok, _ in RESULTS.values())} FAIL")